# Combinatorial AAV payload library: design, screen, and visualise

Designing a gene therapy vector means choosing from a menu of interchangeable
regulatory parts — promoter, transgene, polyadenylation signal. Each combination
must be screened for problem sequences before synthesis. The difficulty is that
some sites are invisible until two specific parts are placed next to each other:
the recognition sequence is split across the junction and present in neither part
alone.

Gen's graph model stores all combinations as paths through a single graph, so one
`search()` call screens every path simultaneously — including every junction.

**Scenario:** An AAV9 vector for limb-girdle muscular dystrophy type 2D (LGMD2D)
caused by loss of α-sarcoglycan (SGCA). The manufacturing protocol linearises the
transfer plasmid with **SbfI** (`CCTGCAGG`) before packaging; any internal SbfI
site would shatter the payload at this step.

- 3 promoters: MCK, CK8e, CAG
- 2 SGCA coding sequences: human codon-optimised, CpG-deoptimised
- 2 polyA signals: bGH, SV40
- **12 combinations** — but some are blocked by a junction-emergent SbfI site

In [ ]:
import pathlib
import tempfile

import gen

WORK_DIR = pathlib.Path(tempfile.mkdtemp())
repo = gen.Repository(str(WORK_DIR))

## Define parts

Each part's `role` in the vector is tracked in a plain Python dict keyed by
part name (`SequencePart` has no metadata field — roles live alongside the
parts, not inside them). The role drives the colour palette when we call
`plot(colors=...)` — all promoters share one colour, all transgenes another —
so the graph's fork-and-rejoin topology is immediately legible.

The fixed ITR flanks are included as single-element columns so every path
through the graph is a complete, self-contained AAV payload.

In [ ]:
sp = gen.SequencePart  # shorthand

# ITR flanks (abbreviated; real AAV2 ITRs are ~145 nt).
itr_left = [
    sp("itr_left", "CACTCCCTCTCTGCGCGCTCGCTCGCTCACTGAGGCCGGGCGACCAAAGGTCGCCCGACGCCCGGGCTTTGCCCGGGCGGCCTCAGTGAGCGAGCGAGCGCGCAGAG"),
]
itr_right = [
    sp("itr_right", "CTCTGCGCGCTCGCTCGCTCACTGAGGCCACCCGACCAAAGGTCGCCCGACGCCCGGGCTTTGCCCGGGCGGCCTCAGTGAGCGAGCGAGCGCGCAGAGAGGGAGTG"),
]

# Three promoters.
# CK8e ends ...CCTGCA — the first six bases of the SbfI octamer.
promoters = [
    sp("MCK",  "AGGCGGGAAGATGGATCCCCTTGAGCAGCTCGAGAGCCTCGAGATGATCCCTTGATCC"),
    sp("CK8e", "TGAAGTGATCCTTGAGCAGCTCGAGAGCCTCGAGATGATCCCTTGGCAGCTAGTCAGCCAGTGCCTGCA"),
    sp("CAG",  "GACATTGATTATTGACTAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCAGATCT"),
]

# Two SGCA coding sequences.
# sgca_human_coopt carries a 7-nt Kozak leader (GGCCACC) before the ATG. The
# leading GG also pairs with the terminal CCTGCA of CK8e to complete the SbfI
# octamer CCTGCAGG.
# sgca_deopt starts directly at ATG.
transgenes = [
    sp("sgca_human_coopt",
       "GGCCACCATGGCGCAGGTCCTGGAGCTGCTGGAGAAGCTGCAGAAGCAGAAGATCGTCATGGACGAGCTGGACGAACTTCAGCTGTGA"),
    sp("sgca_deopt",
       "ATGCAAGTGCTGGAGCTGCTGGAGAAGCTGCAGAAGCAGAAGATCGTCATGGACGAGCTGGACGAGCTTCAGTTGTGA"),
]

# Two polyadenylation signals.
polya = [
    sp("bGH_polyA",
       "TGTATTTGTTTTTTTGTATAGCATAGATGATAATATTTCAGGGCCCAGACATGATAAGATACATTGATGAGTTTGG"),
    sp("SV40_polyA",
       "AGATCTGAATTTTTGTTTTTATTTGTTTTATTTTTTAATTTAAAATAAATATTATTTTTAAATATTATTTTATTTTTTAATTT"),
]

parts_list = [itr_left, promoters, transgenes, polya, itr_right]
n_combinations = 1
for col in parts_list:
    n_combinations *= len(col)
print(f"{' × '.join(str(len(c)) for c in parts_list)} = {n_combinations} combinations")

# Name → role lookup, used for colouring the graph below. import_library()
# creates one stored annotation per part, named after the part.
PART_ROLES = {part.name: role for parts, role in [
    (itr_left, "itr"),
    (promoters, "promoter"),
    (transgenes, "transgene"),
    (polya, "polya"),
    (itr_right, "itr"),
] for part in parts}

## Import the library

`import_library` accepts a list of columns; each column is a list of
`SequencePart` objects. Every combination of one part per column becomes a
distinct path through the resulting graph — 12 paths stored compactly in a
single graph structure.

In [ ]:
repo.import_library("LGMD2D-AAV9", parts_list)

sgs = repo.get_sequence_graphs()
sg = next(s for s in sgs if s.name == "LGMD2D-AAV9")
print(sg)

## Visualise the library graph

The `colors` parameter accepts a callable that receives each stored `Annotation`
and returns a CSS hex colour (or `None` to suppress it). Here we map each
annotation's name back to its role via `PART_ROLES` — all promoters in blue,
all transgenes in green, polyA in amber, and ITRs in grey — so the graph's
fork-and-rejoin topology is immediately legible without reading part names.

In [ ]:
ROLE_COLORS = {
    "itr":       "#aaaaaa",
    "promoter":  "#4393c3",
    "transgene": "#1a9850",
    "polya":     "#e08214",
}

fig = sg.plot(
    rows=18,
    colors=lambda ann: ROLE_COLORS.get(PART_ROLES.get(ann.name)),
)


fig

## Screen the full library for SbfI

A single `search()` call traverses all 12 paths simultaneously, including
every inter-part junction. SbfI (`CCTGCAGG`) would linearise the transfer
plasmid and shatter the payload during packaging.

In [ ]:
sbfi_matches = sg.search("CCTGCAGG", "dna")
print(f"SbfI sites found across the full library: {len(sbfi_matches)}")

## Diagnose: the site is junction-emergent

The match count is surprising — neither CK8e nor sgca_human_coopt carries SbfI
on its own. The recognition sequence is split across their junction:

- CK8e ends with `...CCTGCA` — the first 6 bases of the SbfI octamer
- sgca_human_coopt begins with `GGCCACC...` — the Kozak consensus, supplying `GG`

Together they complete `CCTGCA·GG` = `CCTGCAGG`. This site is invisible to any
PCR screen of isolated parts; it only surfaces when the full junction sequence is
interrogated.

In [ ]:
def count_sbfi(seq: str) -> int:
    site = "CCTGCAGG"
    return sum(1 for i in range(len(seq) - len(site) + 1) if seq[i:i+len(site)] == site)

print("SbfI in individual parts:")
for part in promoters + transgenes:
    n = count_sbfi(part.sequence)
    flag = " ← !" if n else ""
    print(f"  {part.name:<22} {n} site(s){flag}")

print()
junction = promoters[1].sequence + transgenes[0].sequence  # CK8e + sgca_human_coopt
print(f"CK8e ends:               ...{promoters[1].sequence[-12:]}")
print(f"sgca_human_coopt starts:    {transgenes[0].sequence[:12]}...")
print(f"SbfI at CK8e+sgca_human_coopt junction: {count_sbfi(junction)}")

## Highlight the SbfI site on the graph

Navigate to the site and mark it in red. The graph makes the junction context
visible: the site straddles the CK8e→transgene edge — present when those two
nodes are adjacent on a path, absent on every other path through the graph.

In [ ]:
fig2 = sg.plot(
    rows=18,
    colors=lambda ann: ROLE_COLORS.get(PART_ROLES.get(ann.name)),
)

sbfi_matches = sg.search("CCTGCAGG", "dna")
print(f"SbfI sites found across the full library: {len(sbfi_matches)}")

for match in sbfi_matches:
    fig2.show(match, color="#d62728")
fig2

## Export the library as GFA

A combinatorial library is a graph with many valid paths through it, not a
single sequence — `export_fasta` (and `export_genbank`) only capture one
block group's *current path* (whichever combination was inserted most
recently), discarding the rest. `export_gfa` exports the full graph
instead: every part as a shared segment, and every junction between parts
as a link — the complete bubble structure that encodes all 12 combinations,
even though only one of them is recorded as an explicit named path in the
file.

That graph-native structure is the right input for downstream alignment:
a graph-aware aligner such as minigraph2 can align reads or assemblies
against the whole bubble graph, walking any of the 12 candidate constructs,
rather than against one flattened linear sequence at a time.

In [ ]:
out_gfa = str(WORK_DIR / "LGMD2D_AAV9_v1.gfa")
repo.export_gfa(out_gfa)
print(f"Exported to {out_gfa}")
with open(out_gfa) as f:
    lines = f.readlines()
segment_count = sum(1 for l in lines if l.startswith("S\t"))
link_count = sum(1 for l in lines if l.startswith("L\t"))
print(f"{segment_count} segment(s), {link_count} link(s) — the full combinatorial graph")